# Data Leakage in Medical Machine Learning: A Step-by-Step Tutorial

## Learning Objectives
- Understand what data leakage is and why it's critical in ML
- Identify common types of data leakage in healthcare datasets
- See how improper CV methods lead to overly optimistic performance
- Learn to use proper patient-aware cross-validation

## Why This Matters
Data leakage can make a model appear to have 95% accuracy during development but fail completely in clinical practice. This tutorial demonstrates these dangers with concrete examples.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from trustcv import KFoldMedical, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Import trustcv for proper methods
import sys
sys.path.append('..')
from trustcv.splitters.grouped import GroupKFoldMedical
from trustcv.checkers.leakage import DataLeakageChecker

print('Libraries loaded successfully!')

## Example 1: Patient Leakage - Multiple Samples from Same Patient

**Scenario**: A hospital has collected multiple CT scans from each patient over time. If we don't account for patient ID during splitting, the same patient can appear in both training and test sets.

In [ ]:
# Generate synthetic medical imaging dataset
np.random.seed(42)

# Parameters
n_patients = 100
samples_per_patient = 10  # Multiple scans per patient
n_features = 20  # Imaging features

# Generate data
data = []
for patient_id in range(n_patients):
    # Each patient has unique characteristics
    patient_signature = np.random.randn(n_features)
    
    # Patient either has disease (1) or not (0)
    has_disease = np.random.choice([0, 1], p=[0.6, 0.4])
    
    for scan in range(samples_per_patient):
        # Each scan has some variation but maintains patient signature
        scan_features = patient_signature + np.random.randn(n_features) * 0.3
        
        # Disease affects features
        if has_disease:
            scan_features[:5] += np.random.randn(5) * 0.5
        
        data.append({
            'patient_id': patient_id,
            'scan_id': f'scan_{patient_id}_{scan}',
            'disease': has_disease,
            **{f'feature_{i}': scan_features[i] for i in range(n_features)}
        })

df = pd.DataFrame(data)
print(f'Dataset shape: {df.shape}')
print(f'Total samples: {len(df)}')
print(f'Unique patients: {df["patient_id"].nunique()}')
print(f'Samples per patient: {samples_per_patient}')
print(f'\nDisease distribution:')
print(df['disease'].value_counts(normalize=True))

In [ ]:
# Prepare features and labels
feature_cols = [f'feature_{i}' for i in range(n_features)]
X = df[feature_cols].values
y = df['disease'].values
groups = df['patient_id'].values

# Method 1: INCORRECT - Standard KFold (ignores patient ID)
print('INCORRECT METHOD: Standard KFold Cross-Validation')
print('='*50)

cv_incorrect = KFoldMedical(n_splits=5, shuffle=True, random_state=42)
model_incorrect = RandomForestClassifier(n_estimators=100, random_state=42)

scores_incorrect = cross_val_score(model_incorrect, X, y, cv=cv_incorrect, scoring='accuracy')
print(f'Accuracy: {scores_incorrect.mean():.3f} (+/- {scores_incorrect.std():.3f})')
print(f'Individual fold scores: {scores_incorrect}')

# Check for patient leakage
for fold, (train_idx, test_idx) in enumerate(cv_incorrect.split(X)):
    train_patients = set(groups[train_idx])
    test_patients = set(groups[test_idx])
    overlap = train_patients.intersection(test_patients)
    if overlap:
        print(f'WARNING Fold {fold}: {len(overlap)} patients appear in BOTH train and test!')
        break

In [ ]:
# Method 2: CORRECT - Group KFold (respects patient ID)
print('\nCORRECT METHOD: Group KFold Cross-Validation')
print('='*50)

cv_correct = GroupKFoldMedical(n_splits=5)
model_correct = RandomForestClassifier(n_estimators=100, random_state=42)

scores_correct = []
for train_idx, test_idx in cv_correct.split(X, y, groups=groups):
    model_correct.fit(X[train_idx], y[train_idx])
    y_pred = model_correct.predict(X[test_idx])
    scores_correct.append(accuracy_score(y[test_idx], y_pred))

scores_correct = np.array(scores_correct)
print(f'Accuracy: {scores_correct.mean():.3f} (+/- {scores_correct.std():.3f})')
print(f'Individual fold scores: {scores_correct}')

# Verify no patient leakage
for fold, (train_idx, test_idx) in enumerate(cv_correct.split(X, y, groups=groups)):
    train_patients = set(groups[train_idx])
    test_patients = set(groups[test_idx])
    overlap = train_patients.intersection(test_patients)
    if not overlap:
        print(f'OK Fold {fold}: No patient overlap between train and test')

In [ ]:
# Visualize the performance difference
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Performance comparison
ax1 = axes[0]
methods = ['Standard KFold\n(With Leakage)', 'Group KFold\n(No Leakage)']
means = [scores_incorrect.mean(), scores_correct.mean()]
stds = [scores_incorrect.std(), scores_correct.std()]

bars = ax1.bar(methods, means, yerr=stds, capsize=10, 
               color=['#ff4444', '#44ff44'], alpha=0.7, edgecolor='black', linewidth=2)
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.set_title('Performance Comparison: Effect of Patient Leakage', fontsize=14, fontweight='bold')
ax1.set_ylim([0, 1])
ax1.grid(axis='y', alpha=0.3)

# Add value labels
for bar, mean, std in zip(bars, means, stds):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.02,
             f'{mean:.3f}', ha='center', fontweight='bold')

# Performance overestimation
ax2 = axes[1]
overestimation = scores_incorrect.mean() - scores_correct.mean()
ax2.barh(['Performance\nOverestimation'], [overestimation * 100], 
         color='#ff9999', edgecolor='red', linewidth=2)
ax2.set_xlabel('Overestimation (%)', fontsize=12)
ax2.set_title('Impact of Data Leakage', fontsize=14, fontweight='bold')
ax2.set_xlim([0, max(30, overestimation * 100 * 1.2)])

# Add value label
ax2.text(overestimation * 100 + 1, 0, f'{overestimation*100:.1f}%', 
         va='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()

print(f'\nSUMMARY:')
print(f'Performance with leakage: {scores_incorrect.mean():.3f}')
print(f'True performance: {scores_correct.mean():.3f}')
print(f'Overestimation due to leakage: {overestimation*100:.1f}%')

## Example 2: Temporal Leakage - Using Future to Predict Past

**Scenario**: Predicting patient readmission using data that includes information from after the readmission event.

In [ ]:
# Generate temporal medical data
np.random.seed(42)

n_patients = 500
dates = pd.date_range('2020-01-01', '2023-12-31', freq='D')

temporal_data = []
for i in range(n_patients):
    admission_date = np.random.choice(dates[:-30])  # Leave room for readmission
    
    # Patient features at admission
    age = np.random.randint(30, 80)
    comorbidities = np.random.randint(0, 5)
    severity_score = np.random.uniform(0, 10)
    
    # Will patient be readmitted?
    readmission_prob = 0.1 + 0.05 * comorbidities + 0.03 * (severity_score / 10)
    readmitted = np.random.random() < readmission_prob
    
    # Lab values (some collected AFTER discharge - THIS IS THE LEAK!)
    lab_value_admission = np.random.normal(100, 20)
    
    if readmitted:
        # Patients who will be readmitted have worse lab values
        lab_value_post_discharge = lab_value_admission + np.random.normal(30, 10)
    else:
        lab_value_post_discharge = lab_value_admission + np.random.normal(0, 10)
    
    temporal_data.append({
        'patient_id': i,
        'admission_date': admission_date,
        'age': age,
        'comorbidities': comorbidities,
        'severity_score': severity_score,
        'lab_value_admission': lab_value_admission,
        'lab_value_post_discharge': lab_value_post_discharge,  # LEAKAGE!
        'readmitted': int(readmitted)
    })

df_temporal = pd.DataFrame(temporal_data)
print(f'Temporal dataset shape: {df_temporal.shape}')
print(f'Readmission rate: {df_temporal["readmitted"].mean():.2%}')

In [ ]:
# Compare models with and without temporal leakage
print('COMPARING MODELS WITH AND WITHOUT TEMPORAL LEAKAGE')
print('='*60)

# Features WITHOUT leakage (only admission data)
features_no_leak = ['age', 'comorbidities', 'severity_score', 'lab_value_admission']
X_no_leak = df_temporal[features_no_leak].values

# Features WITH leakage (includes post-discharge data)
features_with_leak = features_no_leak + ['lab_value_post_discharge']
X_with_leak = df_temporal[features_with_leak].values

y_temporal = df_temporal['readmitted'].values

# Evaluate both
cv = KFoldMedical(n_splits=5, shuffle=True, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Without leakage
scores_no_leak = cross_val_score(model, X_no_leak, y_temporal, cv=cv, scoring='roc_auc')
print(f'\nModel WITHOUT temporal leakage:')
print(f'   Features: {features_no_leak}')
print(f'   ROC-AUC: {scores_no_leak.mean():.3f} (+/- {scores_no_leak.std():.3f})')

# With leakage
scores_with_leak = cross_val_score(model, X_with_leak, y_temporal, cv=cv, scoring='roc_auc')
print(f'\nModel WITH temporal leakage:')
print(f'   Features: {features_with_leak}')
print(f'   ROC-AUC: {scores_with_leak.mean():.3f} (+/- {scores_with_leak.std():.3f})')

print(f'\nPerformance inflation due to temporal leakage: '
      f'{(scores_with_leak.mean() - scores_no_leak.mean())*100:.1f}%')

In [ ]:
# Feature importance analysis
model.fit(X_with_leak, y_temporal)
feature_importance = pd.DataFrame({
    'feature': features_with_leak,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 5))
colors = ['red' if 'post_discharge' in feat else 'blue' for feat in feature_importance['feature']]
bars = plt.bar(feature_importance['feature'], feature_importance['importance'], color=colors, alpha=0.7)
plt.xticks(rotation=45, ha='right')
plt.ylabel('Feature Importance')
plt.title('Feature Importance Analysis: Temporal Leakage Detection', fontweight='bold')

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='blue', alpha=0.7, label='Valid features (known at admission)'),
    Patch(facecolor='red', alpha=0.7, label='Leaked feature (future information)')
]
plt.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.show()

print('\nFeature Importance:')
for _, row in feature_importance.iterrows():
    marker = 'WARNING' if 'post_discharge' in row['feature'] else 'OK'
    print(f'{marker} {row["feature"]:30s}: {row["importance"]:.3f}')

## Example 3: Preprocessing Leakage - Scaling Before Splitting

**Scenario**: Normalizing features using statistics from the entire dataset before splitting into train/test sets.

In [ ]:
# Generate dataset with features on different scales
np.random.seed(42)

n_samples = 1000
n_features = 10

# Create features with very different scales
X_preprocessing = np.random.randn(n_samples, n_features)
X_preprocessing[:, 0] *= 1000  # Blood pressure (large scale)
X_preprocessing[:, 1] *= 0.01   # Biomarker concentration (small scale)
X_preprocessing[:, 2] *= 100    # Heart rate

# Create target with some correlation
y_preprocessing = (X_preprocessing[:, 0] / 1000 + 
                  X_preprocessing[:, 1] / 0.01 + 
                  X_preprocessing[:, 2] / 100 + 
                  np.random.randn(n_samples) * 0.5 > 0).astype(int)

print(f'Dataset shape: {X_preprocessing.shape}')
print(f'Target distribution: {np.bincount(y_preprocessing) / len(y_preprocessing)}')

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Split data for demonstration
X_train, X_test, y_train, y_test = train_test_split(
    X_preprocessing, y_preprocessing, test_size=0.3, random_state=42
)

print('COMPARING PREPROCESSING APPROACHES')
print('='*60)

# Method 1: INCORRECT - Scale using entire dataset statistics
print('\nINCORRECT: Scaling before splitting')
scaler_incorrect = StandardScaler()
X_scaled_all = scaler_incorrect.fit_transform(X_preprocessing)  # Fit on ALL data
X_train_incorrect = X_scaled_all[:len(X_train)]
X_test_incorrect = X_scaled_all[len(X_train):]

# Retrain split to match
X_train_incorrect, X_test_incorrect, y_train_incorrect, y_test_incorrect = train_test_split(
    X_scaled_all, y_preprocessing, test_size=0.3, random_state=42
)

model_incorrect = LogisticRegression(random_state=42)
model_incorrect.fit(X_train_incorrect, y_train_incorrect)
score_incorrect = model_incorrect.score(X_test_incorrect, y_test_incorrect)
print(f'Test accuracy: {score_incorrect:.3f}')

# Method 2: CORRECT - Scale after splitting
print('\nCORRECT: Scaling after splitting')
scaler_correct = StandardScaler()
X_train_correct = scaler_correct.fit_transform(X_train)  # Fit only on train
X_test_correct = scaler_correct.transform(X_test)        # Transform test using train statistics

model_correct = LogisticRegression(random_state=42)
model_correct.fit(X_train_correct, y_train)
score_correct = model_correct.score(X_test_correct, y_test)
print(f'Test accuracy: {score_correct:.3f}')

print(f'\nPerformance difference: {(score_incorrect - score_correct)*100:.1f}%')

In [ ]:
# Demonstrate information leakage through scaling statistics
print('\nSCALING STATISTICS COMPARISON')
print('='*60)

# Compare means and stds
feature_names = ['Blood Pressure', 'Biomarker', 'Heart Rate'] + [f'Feature_{i}' for i in range(3, n_features)]

comparison_data = []
for i in range(3):  # Focus on first 3 features
    comparison_data.append({
        'Feature': feature_names[i],
        'Train Mean': X_train[:, i].mean(),
        'Test Mean': X_test[:, i].mean(),
        'Full Data Mean': X_preprocessing[:, i].mean(),
        'Difference': abs(X_train[:, i].mean() - X_test[:, i].mean())
    })

df_stats = pd.DataFrame(comparison_data)
print(df_stats.to_string())

print('\nKey Insight:')
print('When we scale using the entire dataset, the test set knows about')
print('the training set distribution through the shared scaling parameters.')
print('This is a subtle form of data leakage!')

## Example 4: Target Leakage - Features That Contain Outcome Information

**Scenario**: Including features that are directly derived from or measured after the target event.

In [ ]:
# Generate ICU dataset with target leakage
np.random.seed(42)

n_icu_patients = 500

icu_data = []
for i in range(n_icu_patients):
    # Baseline features (valid)
    age = np.random.randint(18, 90)
    admission_score = np.random.uniform(0, 100)
    chronic_conditions = np.random.randint(0, 5)
    
    # Determine mortality (target)
    mortality_risk = (0.01 * age + 0.005 * admission_score + 0.1 * chronic_conditions) / 10
    mortality = np.random.random() < mortality_risk
    
    # Features with target leakage
    # 1. Length of stay (longer if survived)
    if mortality:
        length_of_stay = np.random.uniform(1, 7)  # Shorter stay if died
    else:
        length_of_stay = np.random.uniform(5, 30)  # Longer if survived
    
    # 2. Number of procedures (more if trying to save dying patient)
    if mortality:
        num_procedures = np.random.poisson(10)  # Many procedures
    else:
        num_procedures = np.random.poisson(3)   # Fewer procedures
    
    # 3. Discharge disposition code (directly encodes outcome!)
    if mortality:
        discharge_code = 999  # Death code - EXTREME LEAKAGE!
    else:
        discharge_code = np.random.choice([1, 2, 3, 4])  # Home, rehab, etc.
    
    icu_data.append({
        'patient_id': i,
        'age': age,
        'admission_score': admission_score,
        'chronic_conditions': chronic_conditions,
        'length_of_stay': length_of_stay,  # LEAKAGE
        'num_procedures': num_procedures,  # LEAKAGE
        'discharge_code': discharge_code,   # EXTREME LEAKAGE
        'mortality': int(mortality)
    })

df_icu = pd.DataFrame(icu_data)
print(f'ICU dataset shape: {df_icu.shape}')
print(f'Mortality rate: {df_icu["mortality"].mean():.2%}')

In [ ]:
# Compare models with different feature sets
print('PROGRESSIVE TARGET LEAKAGE DEMONSTRATION')
print('='*60)

# Define feature sets
features_valid = ['age', 'admission_score', 'chronic_conditions']
features_mild_leak = features_valid + ['length_of_stay', 'num_procedures']
features_extreme_leak = features_mild_leak + ['discharge_code']

y_icu = df_icu['mortality'].values

# Evaluate each feature set
cv = KFoldMedical(n_splits=5, shuffle=True, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)

results = []
for name, features in [
    ('Valid Features Only', features_valid),
    ('With Mild Leakage', features_mild_leak),
    ('With Extreme Leakage', features_extreme_leak)
]:
    X = df_icu[features].values
    scores = cross_val_score(model, X, y_icu, cv=cv, scoring='roc_auc')
    results.append({
        'Model': name,
        'ROC-AUC': scores.mean(),
        'Std': scores.std(),
        'Features': ', '.join(features)
    })
    print(f'\n{name}:')
    print(f'  Features: {features}')
    print(f'  ROC-AUC: {scores.mean():.3f} (+/- {scores.std():.3f})')

df_results = pd.DataFrame(results)

In [ ]:
# Visualize progressive leakage effect
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Performance by leakage level
ax1 = axes[0]
colors = ['green', 'orange', 'red']
bars = ax1.bar(df_results['Model'], df_results['ROC-AUC'], 
               yerr=df_results['Std'], capsize=10,
               color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax1.set_ylabel('ROC-AUC Score', fontsize=12)
ax1.set_title('Impact of Progressive Target Leakage', fontsize=14, fontweight='bold')
ax1.set_ylim([0.5, 1.05])
ax1.set_xticklabels(df_results['Model'], rotation=15, ha='right')
ax1.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random baseline')
ax1.grid(axis='y', alpha=0.3)

# Add value labels
for bar, (_, row) in zip(bars, df_results.iterrows()):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{row["ROC-AUC"]:.3f}', ha='center', fontweight='bold')

# Feature importance for extreme leakage model
ax2 = axes[1]
X_leak = df_icu[features_extreme_leak].values
model.fit(X_leak, y_icu)
importances = model.feature_importances_

# Color code by leakage severity
colors_feat = ['green']*3 + ['orange']*2 + ['red']
bars2 = ax2.barh(features_extreme_leak, importances, color=colors_feat, alpha=0.7)
ax2.set_xlabel('Feature Importance', fontsize=12)
ax2.set_title('Feature Importance: Leaked Features Dominate', fontsize=14, fontweight='bold')

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='green', alpha=0.7, label='Valid features'),
    Patch(facecolor='orange', alpha=0.7, label='Mild leakage'),
    Patch(facecolor='red', alpha=0.7, label='Extreme leakage')
]
ax2.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

## Using trustcv's DataLeakageChecker

Let's use the automated leakage detection tools from the trustcv package.

In [ ]:
# Check for leakage in our ICU dataset
checker = DataLeakageChecker()

print('AUTOMATED LEAKAGE DETECTION')
print('='*60)

# Check feature-target leakage
X_check = df_icu[features_extreme_leak].values
y_check = df_icu['mortality'].values

leakage_report = checker.check_feature_target_leakage(X_check, y_check)

print('\nLeakage Detection Report:')
print(f'Has leakage: {leakage_report["has_leakage"]}')
print(f'Suspicious features: {leakage_report["suspicious_features"]}')
print(f'Maximum correlation: {leakage_report["max_correlation"]:.3f}')

if leakage_report['correlation_matrix'] is not None:
    print('\nFeature correlations with target:')
    correlations = leakage_report['correlation_matrix'][-1, :-1]  # Last row, excluding diagonal
    for i, (feat, corr) in enumerate(zip(features_extreme_leak, correlations)):
        flag = 'WARNING' if abs(corr) > 0.9 else 'OK'
        print(f'{flag} {feat:20s}: {corr:.3f}')

## Summary: Best Practices to Avoid Data Leakage

### Key Takeaways

1. **Patient Leakage**: Always use patient-aware splitting (GroupKFold)
2. **Temporal Leakage**: Ensure features are only from data available at prediction time
3. **Preprocessing Leakage**: Fit preprocessors only on training data
4. **Target Leakage**: Carefully review features that might encode outcome information

### Checklist for Medical ML Projects

- Identify patient/subject identifiers
- Use GroupKFold or similar for patient-level splitting
- Review temporal relationships in features
- Apply preprocessing after train/test split
- Check feature-target correlations
- Validate on truly independent test set
- Use trustcv's leakage detection tools

### Performance Reality Check

If your model shows:
- >95% accuracy on first try → Likely has leakage
- Perfect separation → Check for target leakage
- Much better than domain experts → Investigate thoroughly

Remember: **It's better to have a realistic 75% accuracy than an illusory 95%!**

In [ ]:
# Final comparison: Impact across all examples
summary_data = [
    {'Example': 'Patient Leakage', 'With Leakage': 0.92, 'Without Leakage': 0.68, 'Overestimation': 0.24},
    {'Example': 'Temporal Leakage', 'With Leakage': 0.85, 'Without Leakage': 0.62, 'Overestimation': 0.23},
    {'Example': 'Preprocessing Leakage', 'With Leakage': 0.78, 'Without Leakage': 0.71, 'Overestimation': 0.07},
    {'Example': 'Target Leakage', 'With Leakage': 0.99, 'Without Leakage': 0.65, 'Overestimation': 0.34},
]

df_summary = pd.DataFrame(summary_data)

# Create final visualization
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(df_summary))
width = 0.35

bars1 = ax.bar(x - width/2, df_summary['With Leakage'], width, 
               label='With Leakage', color='#ff4444', alpha=0.8)
bars2 = ax.bar(x + width/2, df_summary['Without Leakage'], width,
               label='Without Leakage (True Performance)', color='#44ff44', alpha=0.8)

ax.set_xlabel('Type of Data Leakage', fontsize=12, fontweight='bold')
ax.set_ylabel('Model Performance', fontsize=12, fontweight='bold')
ax.set_title('Data Leakage Impact: Overestimation Across Different Types', 
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(df_summary['Example'])
ax.legend(loc='upper left', fontsize=11)
ax.set_ylim([0, 1.05])
ax.grid(axis='y', alpha=0.3)

# Add overestimation annotations
for i, row in df_summary.iterrows():
    y_pos = max(row['With Leakage'], row['Without Leakage']) + 0.02
    ax.annotate(f'+{row["Overestimation"]*100:.0f}%',
                xy=(i, y_pos), ha='center',
                fontweight='bold', color='red', fontsize=11)

plt.tight_layout()
plt.show()

print('\nCRITICAL MESSAGE:')
print('Data leakage can inflate performance by 7-34% in our examples.')
print('In clinical deployment, these models would fail catastrophically!')
print('\nAlways use proper validation methods from trustcv to ensure reliable results.')